# Part 3 — Modernizing the Transformer: A Walkthrough

**Goal of this notebook:** Take the vanilla Transformer block from Part 1 and swap each ingredient for the modern Llama / Mistral / Phi version, so you can read the `.py` files in `part_3/` and understand *why* every line exists.

**What you should know going in:**
- Part 1 (positional encoding, attention, FFN, LayerNorm, residuals) is fresh in your head
- Part 2 (training loop, byte tokenizer, simple causal LM) is fresh in your head
- Comfort with PyTorch tensors and matrix shapes

**What you'll have at the end:**
- Intuition for **RMSNorm** vs LayerNorm
- A clear mental model of **RoPE** (rotary position embedding) — including *why* the rotation is the trick
- The same for **SwiGLU**, **KV cache + RollingKV**, and **GQA** (grouped-query attention)
- A working `GPTModern` you can call, with cached and un-cached generation paths side by side

We trace one concrete sentence — `"I love deep learning"` — through every section, with tiny dimensions so every number fits on screen.


## The Map

Every modern Transformer block has the same six ingredients in the same order — only the *recipes* changed since Part 1. Below is the master diagram you'll see again and again. Each section highlights the ingredient it's about in green, while the rest fade to grey, so you always know *where you are* in the block.

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]
    rms1["3.1 RMSNorm 1"]
    attn["3.5 Modern Attention<br>RoPE + GQA + sliding window + sink + KV cache"]
    add1["+ residual"]
    rms2["3.1 RMSNorm 2"]
    ffn["3.3 SwiGLU FFN"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> rms1 --> attn --> add1
    input --> add1
    add1 --> rms2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style rms1 fill:#e8e8e8,stroke:#bbb,color:#777
    style attn fill:#e8e8e8,stroke:#bbb,color:#777
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style rms2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

**What's new vs Part 1**

| Part 1 | Part 3 | Where to read |
|---|---|---|
| LayerNorm | **RMSNorm** | section 3.1 |
| Sinusoidal / learned PE *added at input* | **RoPE** *applied inside attention* | section 3.2 |
| GELU FFN | **SwiGLU** FFN | section 3.3 |
| MHA (Q,K,V each have `n_head` heads) | + KV cache, **GQA**, sliding window, attention sink | sections 3.4, 3.5 |
| Block stacked N times | same — but with the modern subcomponents | section 3.6 |
| no inference path | full `GPTModern` with cached + nocache generate | section 3.7 |

**Anchor sentence (used everywhere):** `"I love deep learning"` → **T = 4** tokens.
**Tiny model dimensions:** `d_model = 16`, `n_head = 4`, `n_kv_head = 2`, `d_head = 4` (even — RoPE requires even head dim).


## Setup

Run this cell once. It adds `part_3/` to the path so we can `import` from its modules, and sets up common imports.


In [ ]:
import sys, pathlib
# This notebook lives inside part_3/. Add its directory to sys.path so local imports work.
NB_DIR = pathlib.Path().resolve()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

print("Setup complete. Working directory:", NB_DIR)


### Anchor inputs

We'll re-use the same fake "embedded sentence" tensor `x_anchor` across many sections. It pretends that the four words *I*, *love*, *deep*, *learning* have already been tokenized and embedded into `d_model = 16` features each. Real models would learn these vectors during training; we use a fixed random tensor so every section sees the same numbers.


In [ ]:
# Anchor sentence: "I love deep learning"  ->  T = 4 tokens
TOKENS = ["I", "love", "deep", "learning"]
T = len(TOKENS)

# Tiny model hyper-parameters used throughout the notebook
# (d_head MUST be even for RoPE -> we pick d_model=16, n_head=4 -> d_head=4)
B          = 1
d_model    = 16
n_head     = 4
n_kv_head  = 2     # GQA: 4 query heads share 2 KV heads (group size = 2)
d_head     = d_model // n_head    # 4
group_size = n_head // n_kv_head  # 2

# Fake "embedded sentence" tensor we reuse downstream
torch.manual_seed(42)
x_anchor = torch.randn(B, T, d_model)
print("x_anchor shape:", tuple(x_anchor.shape))
print("dimensions: B={}, T={}, d_model={}, n_head={}, n_kv_head={}, d_head={}".format(
    B, T, d_model, n_head, n_kv_head, d_head))


---
## 3.1 RMSNorm

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]
    rms1["3.1 RMSNorm 1"]
    attn["3.5 Modern Attention<br>RoPE + GQA + sliding window + sink + KV cache"]
    add1["+ residual"]
    rms2["3.1 RMSNorm 2"]
    ffn["3.3 SwiGLU FFN"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> rms1 --> attn --> add1
    input --> add1
    add1 --> rms2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style rms1 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style attn fill:#e8e8e8,stroke:#bbb,color:#777
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style rms2 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

Both LayerNorm (Part 1) and RMSNorm normalize *each token vector independently* (across its `d_model` features). LayerNorm subtracts the mean, divides by the std, and adds a learned `(gamma, beta)`. RMSNorm asks: **do we really need the mean-centering and the bias?**

In practice the answer turned out to be *no* — dropping them gives a faster, simpler op with negligible quality difference. Today RMSNorm is the default in Llama, Mistral, Qwen, Phi, etc.

### Math + intuition

|  | LayerNorm | RMSNorm |
|---|---|---|
| step 1 | $\mu = \text{mean}(x)$ | — |
| step 2 | center: $x - \mu$ | — |
| step 3 | $\sigma^2 = \text{var}(x)$ | $r = \sqrt{\text{mean}(x^2) + \varepsilon}$ |
| step 4 | scale: $(x-\mu)/\sqrt{\sigma^2 + \varepsilon}$ | scale: $x / r$ |
| step 5 | affine: $\gamma \odot \hat x + \beta$ | gain: $g \odot \hat x$ |
| params | $2 \cdot d_{model}$ | $d_{model}$ |

**Plain English:** RMSNorm rescales each token vector so its average squared magnitude is $\approx 1$, then multiplies by a learned per-feature gain. No centering, no bias.


### Visualization

We'll feed `x_anchor` (one row per word) through both norms and compare:
- the raw values
- the row means (LayerNorm zeros them; RMSNorm preserves them)
- the row root-mean-squares (both should land at $\approx 1$)


In [ ]:
from rmsnorm import RMSNorm

ln = nn.LayerNorm(d_model)
rn = RMSNorm(d_model)

x = x_anchor.clone()
y_ln  = ln(x)
y_rms = rn(x)

# Stack the three matrices for a side-by-side heatmap
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, mat, title in zip(axes, [x[0], y_ln[0].detach(), y_rms[0].detach()],
                          ["x (input)", "LayerNorm(x)", "RMSNorm(x)"]):
    im = ax.imshow(mat.numpy(), cmap="RdBu_r", vmin=-3, vmax=3)
    ax.set_yticks(range(T)); ax.set_yticklabels(TOKENS)
    ax.set_xticks(range(d_model)); ax.set_xticklabels(range(d_model), fontsize=8)
    ax.set_title(title)
    for i in range(T):
        for j in range(d_model):
            ax.text(j, i, f"{mat[i,j].item():.1f}", ha="center", va="center", fontsize=6)
plt.tight_layout(); plt.show()

print("Row means  (raw):", x[0].mean(dim=-1).tolist())
print("Row means  (LN ):", y_ln[0].mean(dim=-1).detach().tolist(),  "  <- driven to 0")
print("Row means  (RMS):", y_rms[0].mean(dim=-1).detach().tolist(), "  <- *not* zeroed")
print()
print("Row RMS    (raw):", x[0].pow(2).mean(dim=-1).sqrt().tolist())
print("Row RMS    (LN ):", y_ln[0].pow(2).mean(dim=-1).sqrt().detach().tolist())
print("Row RMS    (RMS):", y_rms[0].pow(2).mean(dim=-1).sqrt().detach().tolist())


### Hand-traced cell

Take the first token row of `x_anchor` (the word *I*), call it `x_0`. Then:

$$\text{rms}(x_0) = \sqrt{\tfrac{1}{16}\sum_{j=0}^{15} x_{0j}^2 + \varepsilon} \quad\Rightarrow\quad y_0 = x_0 / \text{rms}(x_0) \cdot g$$

We compute it manually and check it matches the module output.


In [ ]:
x0 = x_anchor[0, 0]                         # (d_model,)
rms_manual = (x0.pow(2).mean() + 1e-8).sqrt()
y0_manual  = x0 / rms_manual * rn.weight

print("rms(x_0)    manually   :", rms_manual.item())
print("RMSNorm(x_0) manually  :", y0_manual.detach().numpy())
print("RMSNorm(x_0) module    :", y_rms[0, 0].detach().numpy())
print("max abs diff           :", (y0_manual - y_rms[0, 0]).abs().max().item())


### Shape trace

| Stage | Shape |
|---|---|
| input `x` | `(B, T, d_model)` = `(1, 4, 16)` |
| `x.pow(2).mean(dim=-1, keepdim=True)` | `(1, 4, 1)` |
| `r = sqrt(... + eps)` | `(1, 4, 1)` |
| `(x / r) * weight` | `(1, 4, 16)` |
| `weight` (learned gain) | `(d_model,)` = `(16,)` |

### Why simpler? (layered)

1. **Surface:** fewer operations. Skipping the `mean` subtraction saves one reduce + one subtract per token.
2. **Practical:** the $\beta$ bias is mostly redundant — the next linear layer can absorb it.
3. **Deep:** mean centering helps with internal covariate shift, but residual connections already keep activations well-conditioned. Once you have residuals + careful init + warmup, the centering is doing very little real work.

**TL;DR:** RMSNorm is "LayerNorm minus the parts that turned out to be load-bearing only on paper".


---
## 3.2 RoPE — Rotary Position Embeddings

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]
    rms1["3.1 RMSNorm 1"]
    attn["3.5 Modern Attention<br>RoPE + GQA + sliding window + sink + KV cache"]
    add1["+ residual"]
    rms2["3.1 RMSNorm 2"]
    ffn["3.3 SwiGLU FFN"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> rms1 --> attn --> add1
    input --> add1
    add1 --> rms2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style rms1 fill:#e8e8e8,stroke:#bbb,color:#777
    style attn fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style rms2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

(Highlight is on **3.5 Modern Attention** because RoPE lives *inside* attention — there's no separate "PE node" anymore.)

### The question

In Part 1 we added a positional vector to the embedding *once*, at the input. RoPE takes a different approach:
> Don't add anything. Instead, **rotate** the Q and K vectors by a position-dependent angle inside attention.

Why? Because rotation has a beautiful algebraic property: it makes the dot product $\langle Q_p, K_q \rangle$ depend only on the *relative offset* $q - p$, which is what attention should care about anyway.

### Math + intuition

For a head dim $D$ (must be even), pick $D/2$ frequencies:

$$\text{inv\_freq}[i] = \frac{1}{10000^{i / D}}, \quad i = 0, 2, 4, \dots, D-2$$

For position $p$ and pair $i$:

$$\theta_{i,p} = p \cdot \text{inv\_freq}[i]$$

We rotate each pair $(x_{2i}, x_{2i+1})$ by $\theta_{i,p}$:

$$\begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

**Plain English:** every feature *pair* gets its own clock hand. Low pair indices have fast clocks (sensitive to small offsets); high pair indices have slow clocks (sensitive to long-range structure). When we then dot-product two RoPE'd vectors, the answer depends only on the angular difference — i.e., the position offset.

### Visualization 1: cos / sin tables


In [ ]:
from rope_custom import RoPECache, apply_rope_single

# Use a slightly larger head_dim here so the visualization has more pairs.
HEAD_DIM_VIZ = 8
MAX_POS_VIZ  = 16

rc = RoPECache(head_dim=HEAD_DIM_VIZ, max_pos=MAX_POS_VIZ)
print("cos shape:", tuple(rc.cos.shape), "  sin shape:", tuple(rc.sin.shape))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, mat, title in zip(axes, [rc.cos.numpy(), rc.sin.numpy()],
                          ["cos[pos, pair_i]", "sin[pos, pair_i]"]):
    im = ax.imshow(mat, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xlabel("pair index i  (each pair = 2 features)")
    ax.set_ylabel("position pos")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


**What you see:** higher pair index `i` (right side) has slower oscillation in the position axis — exactly the geometric-frequency idea from sinusoidal PE in Part 1, just used differently.

### Visualization 2: rotating one feature pair as a clock hand

Take ONE query vector, ONE pair $(x_0, x_1)$, and watch what RoPE does to it as we move it through positions $p = 0, 1, 2, 3, 4$. The pair lives in a 2D plane, so we can plot it.


In [ ]:
# A single pair (x0, x1) -- pick something off-axis for visibility
pair = torch.tensor([1.0, 0.3])

inv_freq_0 = 1.0 / 10000 ** (0 / HEAD_DIM_VIZ)   # for the i=0 (fastest) pair
positions = list(range(5))

fig, ax = plt.subplots(figsize=(5, 5))
ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
colors = plt.cm.viridis(np.linspace(0, 0.9, len(positions)))
for p, c in zip(positions, colors):
    theta = p * inv_freq_0
    rot = torch.tensor([[math.cos(theta), -math.sin(theta)],
                        [math.sin(theta),  math.cos(theta)]])
    vec = rot @ pair
    ax.annotate("", xy=(vec[0], vec[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=c, lw=2))
    ax.text(vec[0]*1.1, vec[1]*1.1, f"p={p}\n\u03B8={theta:.2f}", color=c, fontsize=9)
ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3); ax.set_aspect("equal")
ax.set_title("RoPE rotates the SAME pair by p \u00B7 inv_freq[0]")
plt.show()


**What you see:** the same starting vector (the one with $p=0$) gets rotated counter-clockwise by an angle proportional to position. Different positions land at different angles — that's the position info that the model can pick up via dot products.

### Visualization 3: why the dot product encodes *relative* position

If we apply RoPE to two vectors $Q$ and $K$, the score $\langle \text{RoPE}_p Q, \text{RoPE}_q K \rangle$ depends only on $q - p$. We verify it numerically.


In [ ]:
# Pick a fixed Q and K (with small head_dim and one head for clarity)
torch.manual_seed(0)
q = torch.randn(1, 1, 1, HEAD_DIM_VIZ)   # (B,H,T=1,D)
k = torch.randn(1, 1, 1, HEAD_DIM_VIZ)

def rotated_dot(p, q_pos):
    """Apply RoPE at position p to q, at position q_pos to k, then return the dot product."""
    cos_p, sin_p = rc.get(torch.tensor([p]))
    cos_q, sin_q = rc.get(torch.tensor([q_pos]))
    qr = apply_rope_single(q, cos_p, sin_p)
    kr = apply_rope_single(k, cos_q, sin_q)
    return (qr * kr).sum().item()

# Two settings with the SAME relative offset of +3 should give the same dot product
print("dot(p=0, q=3):", rotated_dot(0, 3))
print("dot(p=2, q=5):", rotated_dot(2, 5))
print("dot(p=7, q=10):", rotated_dot(7, 10))
print()
# Different relative offsets give different dot products
for rel in [0, 1, 2, 4, 8, 12]:
    print(f"relative offset = {rel:2d}  ->  dot = {rotated_dot(0, rel):+.4f}")


**What you see:** the first three values (all with offset $+3$, just at different absolute positions) are identical. The second list shows that *different* relative offsets give *different* dot products — that's the position signal the model learns to use.

### Hand-traced cell calculation

For pair $i = 0$, position $p = 2$, head_dim $D = 8$:

$$\theta_{0,2} = 2 \cdot \frac{1}{10000^{0/8}} = 2 \cdot 1 = 2.0$$

$$\cos(2.0) \approx -0.416, \quad \sin(2.0) \approx 0.909$$

For pair $i = 2$, position $p = 2$:

$$\theta_{2,2} = 2 \cdot \frac{1}{10000^{2/8}} = 2 \cdot 0.1 = 0.2$$

$$\cos(0.2) \approx 0.980, \quad \sin(0.2) \approx 0.199$$


In [ ]:
# Sanity-check the hand calculation against the cache
cos2, sin2 = rc.get(torch.tensor([2]))   # at position 2
print("cos(theta_{0,2}):  manual=-0.416   from cache:", cos2[0, 0].item())
print("sin(theta_{0,2}):  manual= 0.909   from cache:", sin2[0, 0].item())
print()
print("cos(theta_{2,2}):  manual= 0.980   from cache:", cos2[0, 1].item())
print("sin(theta_{2,2}):  manual= 0.199   from cache:", sin2[0, 1].item())


### Why sin AND cos? (layered)

1. **Surface:** they always come together as a 2D rotation matrix — drop one and you'd lose orthogonality.
2. **Practical:** $(\cos, \sin)$ together give a *unique fingerprint* per position. Multiple frequencies stacked together let the model distinguish thousands of positions exactly the way Part 1's sinusoidal PE did.
3. **Deep:** rotating in pairs **is** a linear operation, so a linear projection (Q-, K-projections) downstream can recover any function of "shifted position" it needs. The shift becomes a learnable rotation, and rotations compose to rotations — exactly the algebra attention's dot product wants.

**TL;DR:** sin + cos = 2D rotation matrix; the dot product of two rotated vectors only depends on the *difference* of angles, i.e. *relative* position.

### Shape trace

| Stage | Shape |
|---|---|
| `inv_freq` | `(D/2,)` |
| `cos`, `sin` table | `(max_pos, D/2)` |
| slice for current positions | `(T, D/2)` |
| input `q` (or `k`) | `(B, H, T, D)` |
| `cos.unsqueeze(0).unsqueeze(0)` | `(1, 1, T, D/2)` |
| pair split: `q[..., ::2]`, `q[..., 1::2]` | each `(B, H, T, D/2)` |
| `xr1 = x1*cos - x2*sin`, `xr2 = x1*sin + x2*cos` | each `(B, H, T, D/2)` |
| interleaved output | `(B, H, T, D)` |

### Compare with the .py file


In [ ]:
# Verify our hand-rolled understanding against the actual module behavior.
# Apply RoPE at start_pos=0 to a small q and confirm the shape and that values changed.
q_small = torch.randn(1, 2, T, d_head)   # d_head=4 already even -> good for RoPE
cos_small, sin_small = RoPECache(head_dim=d_head, max_pos=64).get(torch.arange(T))
q_rot = apply_rope_single(q_small, cos_small, sin_small)
print("input  q :", tuple(q_small.shape))
print("output qr:", tuple(q_rot.shape))
print("changed? ", not torch.allclose(q_small, q_rot))


---
## 3.3 SwiGLU FFN

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]
    rms1["3.1 RMSNorm 1"]
    attn["3.5 Modern Attention<br>RoPE + GQA + sliding window + sink + KV cache"]
    add1["+ residual"]
    rms2["3.1 RMSNorm 2"]
    ffn["3.3 SwiGLU FFN"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> rms1 --> attn --> add1
    input --> add1
    add1 --> rms2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style rms1 fill:#e8e8e8,stroke:#bbb,color:#777
    style attn fill:#e8e8e8,stroke:#bbb,color:#777
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style rms2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

The Part 1 FFN was the simplest possible MLP:
```
x  ->  Linear(d -> 4d)  ->  GELU  ->  Linear(4d -> d)
```
SwiGLU asks: *what if the FFN could decide, per position, which features to let through?* It does so via **gating**: two parallel projections, one of which is squashed by a smooth activation and used as a multiplicative mask on the other.

### Math + intuition

$$\text{SwiGLU}(x) = \big( (x W_1) \odot \text{SiLU}(x W_2) \big) W_3$$

with $\text{SiLU}(z) = z \cdot \sigma(z)$ (Swish). Concretely:
- $a = x W_1$  ← "value branch", carries the candidate signal
- $b = \text{SiLU}(x W_2)$  ← "gate branch", per-feature on/off knob in $\approx [-0.28, +\infty)$
- output is $a \odot b$ projected back down by $W_3$

**Plain English:** instead of a fixed nonlinearity, the model learns a *content-dependent* mask that decides how much of each candidate feature survives. GELU/ReLU are blunt instruments by comparison.

### Visualization: SiLU vs GELU side by side


In [ ]:
from swiglu import SwiGLU

zs = torch.linspace(-4, 4, 200)
silu = F.silu(zs)
gelu = F.gelu(zs)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(zs.numpy(), silu.numpy(), label="SiLU (Swish)", lw=2)
ax.plot(zs.numpy(), gelu.numpy(), label="GELU", lw=2, ls="--")
ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
ax.set_title("Activation curves used in Part 1 (GELU) vs Part 3 (SiLU)")
ax.legend(); plt.show()
print("Both look almost identical -> the win comes from gating, not the activation choice.")


### Visualization: gate values on the anchor sentence

We feed `x_anchor` through SwiGLU and visualize the gate `b = SiLU(x W2)` so you can see what it's masking.


In [ ]:
ffn = SwiGLU(dim=d_model, mult=4, dropout=0.0)
ffn.eval()

# Manually compute the gate so we can plot it
with torch.no_grad():
    a = ffn.w1(x_anchor)            # (B,T,4d)
    b = ffn.act(ffn.w2(x_anchor))   # (B,T,4d)
    y = ffn(x_anchor)               # (B,T,d)

print("a shape :", tuple(a.shape))
print("b shape :", tuple(b.shape))
print("y shape :", tuple(y.shape))

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, mat, title in zip(axes, [a[0].numpy(), b[0].numpy()],
                          ["value branch a = x W1", "gate branch b = SiLU(x W2)"]):
    im = ax.imshow(mat, cmap="RdBu_r", aspect="auto", vmin=-1.5, vmax=1.5)
    ax.set_yticks(range(T)); ax.set_yticklabels(TOKENS)
    ax.set_xlabel("hidden dim (4 * d_model = 64)")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


**What you see:** the gate `b` has different patterns per token row — the model can *learn* to pick which of the 48 hidden directions to amplify or suppress for each word.

### Shape trace

| Stage | Shape |
|---|---|
| input `x` | `(B, T, d_model)` |
| `a = x W1` | `(B, T, 4*d_model)` |
| `b = SiLU(x W2)` | `(B, T, 4*d_model)` |
| `a * b` | `(B, T, 4*d_model)` |
| `(a * b) W3` | `(B, T, d_model)` |

### Param count

| Matrix | Shape | # params (mult=4) |
|---|---|---|
| W1 | `(d, 4d)` | $4 d^2$ |
| W2 | `(d, 4d)` | $4 d^2$ |
| W3 | `(4d, d)` | $4 d^2$ |
| total | | $\mathbf{12 d^2}$ |

GELU FFN by comparison has only $W_1, W_2$ for $\mathbf{8 d^2}$. SwiGLU is 50 % heavier — many recipes shrink `mult` from 4 to $\tfrac{8}{3}$ to make total params equal.

### Why gating? (layered)

1. **Surface:** the gate `b` is *per-token, per-feature* — a fixed activation can't be that flexible.
2. **Practical:** SwiGLU consistently wins on perplexity at the same parameter budget, especially past a few hundred million parameters.
3. **Deep:** a gated linear unit is a tiny universal approximator over its inputs *with a multiplicative interaction*. Composed into a deep stack, this gives the network a much richer set of computable functions than a feed-forward MLP with the same params.

**TL;DR:** GELU FFN is a fixed mask; SwiGLU is a learned, content-dependent mask.

### Compare with the .py file


In [ ]:
# Use the actual module on x_anchor and confirm shape preservation.
y2 = ffn(x_anchor)
print("input :", tuple(x_anchor.shape))
print("output:", tuple(y2.shape))


---
## 3.4 KV cache + RollingKV (sliding window + sink)

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]
    rms1["3.1 RMSNorm 1"]
    attn["3.5 Modern Attention<br>RoPE + GQA + sliding window + sink + KV cache"]
    add1["+ residual"]
    rms2["3.1 RMSNorm 2"]
    ffn["3.3 SwiGLU FFN"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> rms1 --> attn --> add1
    input --> add1
    add1 --> rms2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style rms1 fill:#e8e8e8,stroke:#bbb,color:#777
    style attn fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style rms2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

(Highlight stays on **3.5 Modern Attention** because the KV cache lives inside it. Here we focus on the data structure, not where it's used.)

### The question

When a trained model is *generating*, every new token requires re-running attention. A naive implementation recomputes Q, K, V for the entire prefix at every step — $O(T)$ work per step, $O(T^2)$ total. The fix:

> Keep K and V around between steps. Only compute K and V for the **new** token, then concat with the cached past.

This is the **KV cache**. With it, generating $T$ tokens is $O(T)$ instead of $O(T^2)$.

A second problem: for very long generations the cache itself bloats. **RollingKV** keeps only the last `window` tokens, *plus* the first `sink` tokens (the "attention sink" trick).

### Math

After step $t$ we hold:
- $K_{\text{cache}} \in \mathbb{R}^{B \times H_k \times t \times D}$
- $V_{\text{cache}} \in \mathbb{R}^{B \times H_k \times t \times D}$

On step $t+1$ we compute fresh $k, v$ for the new token:
$$k_{\text{new}}, v_{\text{new}} \in \mathbb{R}^{B \times H_k \times 1 \times D}$$

and update:
$$K_{\text{cache}} \leftarrow \text{concat}(K_{\text{cache}}, k_{\text{new}}, \text{dim}=T)$$

If `len > window + sink`, RollingKV crops:
$$K \leftarrow \text{concat}(K[:, :, :\text{sink}],\ K[:, :, -\text{window}:],\ \text{dim}=T)$$

(same for $V$).

### Visualization: cache length over generation steps


In [ ]:
from kv_cache import RollingKV

window = 4
sink   = 2
rk = RollingKV(window=window, sink=sink)

# Run 12 fake steps and track cache length
unbounded = []
bounded   = []
for step in range(12):
    k_new = torch.randn(1, 2, 1, d_head)
    v_new = torch.randn(1, 2, 1, d_head)
    k, v = rk.step(k_new, v_new)
    unbounded.append(step + 1)              # what the cache *would* be without rolling
    bounded.append(k.size(2))               # what RollingKV actually keeps

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(unbounded, label="naive cache (grows linearly)", marker="o", lw=2)
ax.plot(bounded,   label=f"RollingKV(window={window}, sink={sink})", marker="s", lw=2)
ax.axhline(window + sink, color="red", ls="--", lw=0.8, label=f"cap = window+sink = {window+sink}")
ax.set_xlabel("generation step"); ax.set_ylabel("cache length (tokens)")
ax.set_title("KV cache memory: bounded vs unbounded")
ax.legend(); ax.grid(True, alpha=0.3); plt.show()


**What you see:** the bounded cache plateaus at `window + sink = 6`, while the naive cache grows forever.

### Visualization: which positions remain in cache?


In [ ]:
# Restart and record the position indices kept in the cache after each step.
rk = RollingKV(window=window, sink=sink)
kept = []
for step in range(12):
    k_new = torch.full((1, 2, 1, d_head), float(step))   # mark with the step number
    v_new = torch.full((1, 2, 1, d_head), float(step))
    k, v = rk.step(k_new, v_new)
    # Each entry's first feature value tells us which step it was added at
    positions = k[0, 0, :, 0].tolist()
    kept.append([int(p) for p in positions])

# Build a (T_steps, max_keep) "kept positions" image
T_steps = len(kept)
max_keep = window + sink
img = np.full((T_steps, max_keep), np.nan)
for i, row in enumerate(kept):
    for j, p in enumerate(row):
        img[i, j] = p

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(img, cmap="viridis", aspect="auto")
ax.set_xlabel("slot index in cache (left = sink, right = recent tail)")
ax.set_ylabel("generation step")
ax.set_title("RollingKV: which token positions survive in the cache")
for i in range(T_steps):
    for j in range(max_keep):
        if not np.isnan(img[i, j]):
            ax.text(j, i, f"{int(img[i,j])}", ha="center", va="center", color="white", fontsize=8)
plt.colorbar(im, ax=ax, label="original position"); plt.show()


**What you see:** the leftmost two columns are *frozen* on positions 0 and 1 — those are the **attention sinks**. The rest of the columns slide forward to track the most recent four tokens.

### Why a sink? (layered)

1. **Surface:** the very first tokens get a disproportionate share of attention mass; throwing them away makes downstream tokens panic.
2. **Practical:** Xiao et al. (2023) showed that *just* keeping the first 4 tokens in the cache restores quality during streaming generation almost completely.
3. **Deep:** softmax attention has to allocate its mass *somewhere*. If a token has nothing else to attend to, it dumps mass on whatever is leftmost in the cache. Pre-training fixes those positions to be position 0/1/..., so cropping them shifts the "garbage attention" to something that wasn't an anchor — and the whole layer destabilises.

**TL;DR:** the cache is the bottleneck for long-context inference; sink + sliding window let it stay bounded *without* degrading quality.

### Shape trace

| Stage | Shape |
|---|---|
| K, V at step `t`           | `(B, n_kv_head, t, d_head)` |
| `k_new, v_new`             | `(B, n_kv_head, 1, d_head)` |
| after concat               | `(B, n_kv_head, t+1, d_head)` |
| after RollingKV crop       | `(B, n_kv_head, ≤ window+sink, d_head)` |


---
## 3.5 Modern Attention — putting it all together

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]
    rms1["3.1 RMSNorm 1"]
    attn["3.5 Modern Attention<br>RoPE + GQA + sliding window + sink + KV cache"]
    add1["+ residual"]
    rms2["3.1 RMSNorm 2"]
    ffn["3.3 SwiGLU FFN"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> rms1 --> attn --> add1
    input --> add1
    add1 --> rms2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style rms1 fill:#e8e8e8,stroke:#bbb,color:#777
    style attn fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style rms2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

Now we combine RoPE (3.2), KV cache + sliding window + sink (3.4), and **GQA** (Grouped-Query Attention) into one attention module. The new ingredient here is GQA — the rest is review.

### GQA in one sentence

| | Q heads | K/V heads | who shares with whom |
|---|---|---|---|
| MHA (Part 1) | 4 | 4 | 1-to-1 |
| MQA          | 4 | 1 | all queries share one K/V pair |
| **GQA**      | 4 | 2 | each K/V head is shared by `group_size = 2` query heads |

We pick GQA because:
- it cuts the K/V cache by `group_size`x — and the K/V cache is the *memory bandwidth* bottleneck during inference;
- it loses very little quality vs full MHA.

### Visualization: GQA grouping (n_head=4, n_kv_head=2)


```mermaid
graph TD
    subgraph Queries
      q0["Q head 0"]
      q1["Q head 1"]
      q2["Q head 2"]
      q3["Q head 3"]
    end
    subgraph KV
      kv0["KV head 0"]
      kv1["KV head 1"]
    end
    q0 --> kv0
    q1 --> kv0
    q2 --> kv1
    q3 --> kv1
    style q0 fill:#d6eaf8,stroke:#2874a6,color:#000
    style q1 fill:#d6eaf8,stroke:#2874a6,color:#000
    style q2 fill:#fdebd0,stroke:#b35900,color:#000
    style q3 fill:#fdebd0,stroke:#b35900,color:#000
    style kv0 fill:#d6eaf8,stroke:#2874a6,stroke-width:3px,color:#000
    style kv1 fill:#fdebd0,stroke:#b35900,stroke-width:3px,color:#000
```

Query heads 0 and 1 share KV head 0 (blue); query heads 2 and 3 share KV head 1 (orange). At attention time we `repeat_interleave` the KV heads back to 4 to match the queries — but the *cache* still only stores 2 heads, saving ~50% of memory.

### Shape trace through one forward pass (B=1, T=4, no cache)

| Stage | Shape |
|---|---|
| input `x`             | `(1, 4, 16)` |
| `Wq(x)`               | `(1, 4, 16)`  = `(B, T, n_head*d_head)` |
| `Wk(x)`, `Wv(x)`      | `(1, 4, 8)`   = `(B, T, n_kv_head*d_head)` |
| `q.view + transpose`  | `(1, 4, 4, 4)`  = `(B, n_head, T, d_head)` |
| `k,v.view + transpose`| `(1, 2, 4, 4)`  = `(B, n_kv_head, T, d_head)` |
| RoPE applied to q, k  | same shapes |
| `repeat_interleave(2)`| `(1, 4, 4, 4)`  expanded to match Q heads |
| sdpa output           | `(1, 4, 4, 4)` |
| merge heads           | `(1, 4, 16)` |
| `Wo(...)`             | `(1, 4, 16)` |


### Live walkthrough on the anchor sentence


In [ ]:
from attn_modern import CausalSelfAttentionModern

attn = CausalSelfAttentionModern(
    n_embd=d_model,
    n_head=n_head,
    n_kv_head=n_kv_head,
    rope=True,
    max_pos=64,
)
attn.eval()

with torch.no_grad():
    y, kv = attn(x_anchor, kv_cache=None, start_pos=0)

print("input  x  :", tuple(x_anchor.shape))
print("output y  :", tuple(y.shape))
print("KV cache K:", tuple(kv.k.shape), "  (note: only n_kv_head =", n_kv_head, "K heads)")
print("KV cache V:", tuple(kv.v.shape))


### Generation step: feed past cache + a single new token


In [ ]:
# Now pretend we're generating. Feed in just one new token (T=1) with the cache from above.
new_token = torch.randn(B, 1, d_model)

with torch.no_grad():
    y2, kv2 = attn(new_token, kv_cache=kv, start_pos=T)   # start_pos = T because we already saw T tokens

print("new input shape   :", tuple(new_token.shape))
print("output shape      :", tuple(y2.shape))
print("KV cache K shape  :", tuple(kv2.k.shape), " (grew from", kv.k.size(2), "to", kv2.k.size(2), "tokens)")


### Sliding window + sink in action

Let's run a longer pretend generation and watch the cache stay bounded once we set `sliding_window` and `attention_sink`.


In [ ]:
attn_bounded = CausalSelfAttentionModern(
    n_embd=d_model,
    n_head=n_head,
    n_kv_head=n_kv_head,
    rope=True,
    max_pos=64,
    sliding_window=4,
    attention_sink=2,
)
attn_bounded.eval()

# Prime with the anchor sentence
with torch.no_grad():
    _, kv = attn_bounded(x_anchor, kv_cache=None, start_pos=0)
print(f"after priming with T={T:2d} tokens, cache len = {kv.k.size(2)}")

# Generate 10 more, single tokens at a time
with torch.no_grad():
    for step in range(10):
        nt = torch.randn(B, 1, d_model)
        _, kv = attn_bounded(nt, kv_cache=kv, start_pos=T+step)
        print(f"after step {step+1:2d}, cache len = {kv.k.size(2)}  (bounded by {attn_bounded.sliding_window + attn_bounded.attention_sink})")


**What you see:** once the cache exceeds `window + sink = 6`, it stops growing — exactly the behavior we proved earlier with RollingKV.

### Compare with the .py file

Open `attn_modern.py` in your editor. The forward pass roughly matches:
1. project Q (n_head heads), K, V (n_kv_head heads each)
2. apply RoPE to Q, K
3. concat past cache (if any)
4. crop to sink + sliding window
5. expand K, V from `n_kv_head` to `n_head` by `repeat_interleave`
6. `F.scaled_dot_product_attention(..., is_causal=...)`
7. merge heads, output projection


---
## 3.6 Modern Block

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]
    rms1["3.1 RMSNorm 1"]
    attn["3.5 Modern Attention<br>RoPE + GQA + sliding window + sink + KV cache"]
    add1["+ residual"]
    rms2["3.1 RMSNorm 2"]
    ffn["3.3 SwiGLU FFN"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> rms1 --> attn --> add1
    input --> add1
    add1 --> rms2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style rms1 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style attn fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style add1 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style rms2 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ffn fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style add2 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

The whole block is highlighted now: every node is the modern version of its Part 1 counterpart.

### The question

The block structure (pre-norm, two residuals) is identical to Part 1; only the *components inside* have been swapped. So this section is mostly a sanity-check on shape preservation and parameter count.

### Math

$$x \leftarrow x + \text{ModernAttn}(\text{RMSNorm}(x))$$
$$x \leftarrow x + \text{SwiGLU}(\text{RMSNorm}(x))$$

That's the whole forward pass.


In [ ]:
from block_modern import TransformerBlockModern

block = TransformerBlockModern(
    n_embd=d_model,
    n_head=n_head,
    n_kv_head=n_kv_head,
    use_rmsnorm=True,
    use_swiglu=True,
    rope=True,
    max_pos=64,
)
block.eval()

with torch.no_grad():
    y, kv = block(x_anchor, kv_cache=None, start_pos=0)

print("input  x :", tuple(x_anchor.shape))
print("output y :", tuple(y.shape))
print("KV cache :", tuple(kv.k.shape), "k", tuple(kv.v.shape), "v")

# Param count
total = sum(p.numel() for p in block.parameters())
by_name = {n: p.numel() for n, p in block.named_parameters()}
print()
print(f"Total params in this block: {total:,}")
for name, n in by_name.items():
    print(f"  {name:30s} -> {n:6,}")


### Shape trace

| Stage | Shape |
|---|---|
| input `x`                       | `(B, T, d_model)` |
| `ln1(x)` (RMSNorm)              | `(B, T, d_model)` |
| `attn(...)`                     | `(B, T, d_model)` |
| `x + a` (residual 1)            | `(B, T, d_model)` |
| `ln2(x)` (RMSNorm)              | `(B, T, d_model)` |
| `ffn(...)` (SwiGLU)             | `(B, T, d_model)` |
| `x + ffn(...)` (residual 2)     | `(B, T, d_model)` |
| output                          | `(B, T, d_model)` |

### Compare with the .py file

`block_modern.py` is just five lines of forward:
```python
a, kv_cache = self.attn(self.ln1(x), kv_cache=kv_cache, start_pos=start_pos)
x = x + a
x = x + self.ffn(self.ln2(x))
return x, kv_cache
```


---
## 3.7 GPTModern — full model + generation comparison

```mermaid
graph TD
    ids["Token IDs (B, T)"]
    tok["Token Embedding (B, T, d_model)"]
    blk1["Block 1 (3.6)"]
    blk2["Block 2 (3.6)"]
    blkN["Block N (3.6)"]
    lnf["ln_f (Identity if RMSNorm else LayerNorm)"]
    head["LM head (Linear d_model -> vocab)"]
    out["Logits (B, T, vocab)"]

    ids --> tok --> blk1 --> blk2 --> blkN --> lnf --> head --> out

    style ids fill:#d5f5e3,stroke:#28a745,stroke-width:2px,color:#000
    style tok fill:#d5f5e3,stroke:#28a745,stroke-width:2px,color:#000
    style blk1 fill:#fff3c4,stroke:#b38600,color:#000
    style blk2 fill:#fff3c4,stroke:#b38600,color:#000
    style blkN fill:#fff3c4,stroke:#b38600,color:#000
    style lnf fill:#d6eaf8,stroke:#2874a6,color:#000
    style head fill:#e8daef,stroke:#6c3483,color:#000
    style out fill:#fdebd0,stroke:#b35900,stroke-width:2px,color:#000
```

### The question

What's left is gluing N blocks together with a token embedding and a LM head. There's **no positional embedding** — RoPE inside each attention layer handles position.

For inference we ship two paths so we can prove the cache is correct:

| Method | What it does | Cost per new token |
|---|---|---|
| `generate`         | uses KV cache + sliding window + sink | $O(1)$ tokens to attend over (bounded) |
| `generate_nocache` | recomputes the cropped window every step | $O(\text{window})$ |

### Build a tiny untrained model and run both


In [ ]:
from model_modern import GPTModern
from tokenizer import ByteTokenizer

torch.manual_seed(0)
tok = ByteTokenizer()
model = GPTModern(
    vocab_size=tok.vocab_size,
    block_size=128,
    n_layer=2,
    n_head=4,
    n_embd=64,
    n_kv_head=2,
    use_rmsnorm=True,
    use_swiglu=True,
    rope=True,
    max_pos=4096,
    sliding_window=32,
    attention_sink=2,
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"GPTModern params: {n_params:,}")


### Generate — same prompt, both paths


In [ ]:
import time

prompt = torch.tensor([[10, 72, 105]], dtype=torch.long)   # arbitrary 3 byte ids
N = 60

# Cached generation
t0 = time.time()
with torch.no_grad():
    out_cached = model.generate(prompt.clone(), max_new_tokens=N, temperature=0.0, top_k=50)
t_cached = time.time() - t0

# Nocache generation
t0 = time.time()
with torch.no_grad():
    out_nocache = model.generate_nocache(prompt.clone(), max_new_tokens=N, temperature=0.0, top_k=50)
t_nocache = time.time() - t0

print(f"cached  : {t_cached*1000:7.1f} ms   shape {tuple(out_cached.shape)}")
print(f"nocache : {t_nocache*1000:7.1f} ms   shape {tuple(out_nocache.shape)}")
print(f"speedup : {t_nocache / max(t_cached, 1e-9):.2f}x")


(The output text will be gibberish — the model is untrained. We're only checking timings.)

### Visualization: per-step inference cost

We instrument `generate` to record how long each step takes, comparing cache vs no-cache.


In [ ]:
def time_per_step(generate_fn, N=40):
    times = []
    idx = prompt.clone()
    with torch.no_grad():
        kvs = [None] * len(model.blocks) if generate_fn == "cached" else None
        for step in range(N):
            t0 = time.time()
            if generate_fn == "cached":
                idx_cond = idx[:, -model.block_size:] if kvs[0] is None else idx[:, -1:]
                start_pos = 0 if kvs[0] is None else kvs[0].k.size(2)
                logits, _, kvs = model(idx_cond, kv_cache_list=kvs, start_pos=start_pos)
            else:
                idx_cond = idx[:, -model.block_size:]
                start_pos = idx.size(1) - idx_cond.size(1)
                logits, _, _ = model(idx_cond, kv_cache_list=None, start_pos=start_pos)
            next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_id], dim=1)
            times.append((time.time() - t0) * 1000.0)
    return times

t_cached_steps  = time_per_step("cached")
t_nocache_steps = time_per_step("nocache")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_cached_steps,  marker="o", label="cached", lw=1)
ax.plot(t_nocache_steps, marker="s", label="nocache (recomputes)", lw=1)
ax.set_xlabel("generation step"); ax.set_ylabel("step time (ms)")
ax.set_title("Per-step inference cost: cached vs no-cache")
ax.legend(); ax.grid(alpha=0.3); plt.show()


**What you see:**
- `nocache` step time grows with the number of tokens until it hits the `block_size` / sliding window cap, then plateaus.
- `cached` step time stays roughly flat — only the *new* token is processed, plus a fixed-size cache lookup.

### Closing the loop with the master diagram
```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]
    rms1["3.1 RMSNorm 1"]
    attn["3.5 Modern Attention<br>RoPE + GQA + sliding window + sink + KV cache"]
    add1["+ residual"]
    rms2["3.1 RMSNorm 2"]
    ffn["3.3 SwiGLU FFN"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> rms1 --> attn --> add1
    input --> add1
    add1 --> rms2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style rms1 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style attn fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style add1 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style rms2 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ffn fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style add2 fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

You now have everything needed to read every file in `part_3/`:

- [rmsnorm.py](rmsnorm.py) — section 3.1
- [rope_custom.py](rope_custom.py) — section 3.2
- [swiglu.py](swiglu.py) — section 3.3
- [kv_cache.py](kv_cache.py) — section 3.4
- [attn_modern.py](attn_modern.py) — section 3.5
- [block_modern.py](block_modern.py) — section 3.6
- [model_modern.py](model_modern.py) — section 3.7
- [demo_generate.py](demo_generate.py) — runs the comparison from section 3.7 as a script

### What's next

Part 4 trains a model with these modern components on real text, using BPE tokenization, AMP, gradient accumulation, and a warmup-cosine learning-rate schedule. Everything from this notebook (RMSNorm, RoPE, SwiGLU, GQA, KV cache, sliding window) carries forward.
